In [145]:
import json
import os
import tempfile
from sklearn.metrics import accuracy_score, classification_report
import random
import requests
import statistics
import re
import nltk
from nltk import word_tokenize, pos_tag

In [54]:
#system prompt

generic_prompt = f"""You are an expert academic reviewer that provides actionable feedback on the paper contents provided in the user prompt. Present your feedback as a list of bullet points.
"""

#, such as "Clarify X", "Provide evidence for Y", or "Reorganize Z" for example.
specific_prompt = f"""You are an expert academic reviewer that provides actionable feedback on the paper contents provided in the user prompt. This means your feedback can be immediately implemented by the author to improve the paper. Frame each point as a specific, concrete suggestion or revision command."""

In [82]:
def build_prompt(paper):
  metadata = paper.get('metadata') #metadata dictionary that contains the actual contents of the paper
  content_list = metadata.get('sections')
  #print(content_list)
  #print(content_list[4].get('text'))
  #print(type(content_list[4]))

  paper_content = str(metadata.get('sections'))
  prompt = f"""Paper content: {paper_content}
  Provide your suggestions for the author to improve the paper. """

  return prompt


In [ ]:
def model_forecasting(model, system_prompt, prompt):
    #print(prompt)
    # Send request to Ollama

    res = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": model, #llama3.2:3b , "qwen3:latest"
            "system": system_prompt,
            "prompt": prompt, 
            "stream": False
            }
    )
    result = res.json()
    return result

In [74]:
def give_feedback(pdf_path, system_prompt, results):
    with open(pdf_path, 'r') as f1:
        paper = json.load(f1) #json file contents for one research paper

    prompt = build_prompt(paper)
    model = "llama3.2:latest"
    output = model_forecasting(model, system_prompt, prompt)
    json_response = output["response"]
    #print(json_response)
    results[paper.get("name")] = {
        "review": json_response
    }
    return results

In [83]:
pdf_path = "C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\acl_2017\\train\\parsed_pdfs\\699.pdf.json"
review_path = "C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\acl_2017\\train\\reviews\\699.json"
results = {}
results = give_feedback(pdf_path, specific_prompt, results)
print(results)

{'699.pdf': {'review': 'Here are some suggestions for the author to improve the paper:\n\n1. **Provide more context**: The paper assumes that readers are familiar with keyphrase extraction and the encoder-decoder model. Adding a brief introduction or background section to explain these concepts would help readers who are not familiar with them.\n2. **Improve the organization**: The paper jumps abruptly between different sections (e.g., from introducing the problem to presenting experimental results). Consider breaking up the paper into clearer sections, such as an introduction, literature review, methodology, and discussion.\n3. **Use more formal language**: While the paper is well-written overall, there are some informal phrases and sentences that could be rephrased for a more academic tone (e.g., "We expect the models to be able to learn universal language features" -> "We hypothesize that our model can generalize to other domains by learning generalizable linguistic patterns").\n4. 

In [56]:
def generic_sample(dir_path, sample_size, output_path):
    paper_names = os.listdir(dir_path)
    results = {}
    for i in range(0, sample_size): 
        pdf_path = os.path.join(dir_path, paper_names[i])
        results = give_feedback(pdf_path, generic_prompt, results)
    
    with open(output_path,'a') as f3:
        json.dump(results,f3)


In [57]:
def specific_sample(dir_path, sample_size, output_path):
    paper_names = os.listdir(dir_path)
    results = {}
    for i in range(0, sample_size): 
        pdf_path = os.path.join(dir_path, paper_names[i])
        results = give_feedback(pdf_path, specific_prompt, results)
    
    with open(output_path,'a') as f3:
        json.dump(results,f3)

In [158]:
dir_path = "C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\iclr_2017\\train\\parsed_pdfs"
generic_output_path = "C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\dtais_summer\\generic_iclr_100.json"
specific_output_path = "C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\dtais_summer\\specific_iclr_100.json"
sample_size = 100

In [159]:
generic_sample(dir_path, sample_size, generic_output_path)
specific_sample(dir_path, sample_size, specific_output_path)

# Evaluation

In [154]:

def extract_numbered_suggestions(text):
    """
    Extracts all text blocks between \n\d+. and the next \n\d+. or end of text.
    Handles numbered suggestions like '1. ...', '2. ...'
    """
    pattern = r"\n\d+\.\s+(.*?)(?=\n\d+\.|\Z)"
    matches = re.findall(pattern, text, re.DOTALL)
    # Clean and strip each suggestion
    cleaned_matches = [m.strip() for m in matches if m.strip()]
    print(cleaned_matches)
    return cleaned_matches

    #return "\n".join(suggestions).strip()

def contains_verb(text):
    """Returns True if the text contains at least one verb."""
    tokens = word_tokenize(text)
    tags = pos_tag(tokens)
    return any(tag.startswith("VB") for _, tag in tags)  # VB, VBD, VBG, etc.


def check_verbs(response_file):

    with open(response_file, "r") as f:
        results = json.load(f)
    # Store actionable scores
    actionable_scores = {}

    for paper_name, content in results.items():
        review = content['review']
        suggestions = extract_numbered_suggestions(review)
        total_points = len(suggestions)
        points_with_verb = sum(contains_verb(point) for point in suggestions)

        score = round(points_with_verb / total_points, 2) if total_points else 0.0
        
        actionable_scores[paper_name] = {
            "actionable_score": round(score, 2)
        }

    # Print or save results
    for paper, score_data in actionable_scores.items():
        print(f"{paper}: {score_data}")
    #check for verb word list

<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
C:\Users\G34371231\AppData\Local\Temp\ipykernel_16228\2091896330.py:2: SyntaxWarning: invalid escape sequence '\d'
  """


In [132]:
# coding=utf8
# the above tag defines encoding for this document and is for Python 2.x compatibility

import re

regex1 = r"\\n\d+\.\s+(.*?)(?=\\n\d+\.|\Z)"
regex2 = r"\d+\.\s+(.*?)(?=\d|\Z)"
test_str = "Here are some suggestions for the author to improve the paper:\\n\\n1. **Provide more context**: The paper assumes a high level of familiarity with formal verification, programming languages, and algorithms. It would be helpful to provide more context about the background and relevance of the research.\\n2. **Define key terms**: The paper uses technical terms like \\\"base cases,\\\" \\\"reduction rules,\\\" and \\\"verification set\\\" without explanation. Defining these terms and providing explanations would help readers understand the paper better.\\n3. **Organize the content**: The paper jumps between different topics, such as formal verification, programming languages, and algorithms. Organizing the content into logical sections or chapters would make it easier to follow.\\n4. **Provide more details about the implementation**: While the paper mentions that the authors implemented a variety of algorithms using different programming languages, it would be helpful to provide more details about the implementation, such as the language choices, data structures used, and any notable challenges encountered.\\n5. **Discuss the limitations of the current approach**: The paper assumes that the current approach is complete and optimal. It would be helpful to discuss potential limitations or areas for future research.\\n6. **Compare with existing work**: The paper mentions existing work on formal verification and programming languages, but it would be helpful to provide a more detailed comparison with the existing literature.\\n7. **Use clear and concise notation**: The paper uses some specialized notation that may not be immediately clear to readers. Using clear and concise notation would help readers understand the paper better.\\n8. **Provide visual aids**: The paper is dense with technical information, which can make it difficult to follow. Providing visual aids like diagrams or flowcharts could help illustrate key concepts.\\n9. **Address potential criticisms**: The paper assumes that its approach is correct and optimal without considering potential criticisms. Addressing potential criticisms and counterarguments would strengthen the paper.\\n10. **Consider a more detailed evaluation**: The paper mentions that it has been verified on a limited set of input problems, but it would be helpful to provide more details about the evaluation process and the results obtained.\\n\\nSome additional suggestions:\\n\\n* Consider providing a more detailed explanation of the LSTM model used in the verification process.\\n* Discuss potential applications or use cases for the formal verification approach presented in the paper.\\n* Provide more information about the training data used to train the models, such as the size and characteristics of the dataset.\\n* Consider including more examples or case studies to illustrate the effectiveness of the formal verification approach."


iclr_str = "Here are some suggestions for the author to improve the paper:\n\n1. **Provide more context**: The paper assumes a high level of familiarity with formal verification, programming languages, and algorithms. It would be helpful to provide more context about the background and relevance of the research.\n2. **Define key terms**: The paper uses technical terms like \"base cases,\" \"reduction rules,\" and \"verification set\" without explanation. Defining these terms and providing explanations would help readers understand the paper better.\n3. **Organize the content**: The paper jumps between different topics, such as formal verification, programming languages, and algorithms. Organizing the content into logical sections or chapters would make it easier to follow.\n4. **Provide more details about the implementation**: While the paper mentions that the authors implemented a variety of algorithms using different programming languages, it would be helpful to provide more details about the implementation, such as the language choices, data structures used, and any notable challenges encountered.\n5. **Discuss the limitations of the current approach**: The paper assumes that the current approach is complete and optimal. It would be helpful to discuss potential limitations or areas for future research.\n6. **Compare with existing work**: The paper mentions existing work on formal verification and programming languages, but it would be helpful to provide a more detailed comparison with the existing literature.\n7. **Use clear and concise notation**: The paper uses some specialized notation that may not be immediately clear to readers. Using clear and concise notation would help readers understand the paper better.\n8. **Provide visual aids**: The paper is dense with technical information, which can make it difficult to follow. Providing visual aids like diagrams or flowcharts could help illustrate key concepts.\n9. **Address potential criticisms**: The paper assumes that its approach is correct and optimal without considering potential criticisms. Addressing potential criticisms and counterarguments would strengthen the paper.\n10. **Consider a more detailed evaluation**: The paper mentions that it has been verified on a limited set of input problems, but it would be helpful to provide more details about the evaluation process and the results obtained.\n\nSome additional suggestions:\n\n* Consider providing a more detailed explanation of the LSTM model used in the verification process.\n* Discuss potential applications or use cases for the formal verification approach presented in the paper.\n* Provide more information about the training data used to train the models, such as the size and characteristics of the dataset.\n* Consider including more examples or case studies to illustrate the effectiveness of the formal verification approach."
matches = re.findall(regex2, test_str, re.MULTILINE)
print(iclr_str)
print(test_str)

Here are some suggestions for the author to improve the paper:

1. **Provide more context**: The paper assumes a high level of familiarity with formal verification, programming languages, and algorithms. It would be helpful to provide more context about the background and relevance of the research.
2. **Define key terms**: The paper uses technical terms like "base cases," "reduction rules," and "verification set" without explanation. Defining these terms and providing explanations would help readers understand the paper better.
3. **Organize the content**: The paper jumps between different topics, such as formal verification, programming languages, and algorithms. Organizing the content into logical sections or chapters would make it easier to follow.
4. **Provide more details about the implementation**: While the paper mentions that the authors implemented a variety of algorithms using different programming languages, it would be helpful to provide more details about the implementatio

In [155]:
check_verbs(generic_output_path)

['**Provide more context**: The paper assumes a high level of familiarity with formal verification, programming languages, and algorithms. It would be helpful to provide more context about the background and relevance of the research.', '**Define key terms**: The paper uses technical terms like "base cases," "reduction rules," and "verification set" without explanation. Defining these terms and providing explanations would help readers understand the paper better.', '**Organize the content**: The paper jumps between different topics, such as formal verification, programming languages, and algorithms. Organizing the content into logical sections or chapters would make it easier to follow.', '**Provide more details about the implementation**: While the paper mentions that the authors implemented a variety of algorithms using different programming languages, it would be helpful to provide more details about the implementation, such as the language choices, data structures used, and any no

In [181]:
import json
import re
from nltk import word_tokenize, pos_tag

def extract_numbered_suggestions(text):
    """
    Extracts all text blocks between \n\d+. and the next \n\d+. or end of text.
    Handles numbered suggestions like '1. ...', '2. ...'
    """
    pattern = r"\n\d+\.\s+(.*?)(?=\n\d+\.|\Z)"
    matches = re.findall(pattern, text, re.DOTALL)
    cleaned_matches = [m.strip() for m in matches if m.strip()]
    return cleaned_matches

def contains_verb(text):
    """Returns True if the text contains at least one verb."""
    tokens = word_tokenize(text)
    tags = pos_tag(tokens)
    return any(tag.startswith("VB") for _, tag in tags)  # VB, VBD, VBG, etc.

def compute_actionable_score(review_text):
    suggestions = extract_numbered_suggestions(review_text)
    total_points = len(suggestions)
    points_with_verb = sum(contains_verb(point) for point in suggestions)
    return round(points_with_verb / total_points, 2) if total_points else 0.0

def compare_generic_specific(generic_file, specific_file):
    with open(generic_file, "r") as f:
        generic_data = json.load(f)
    with open(specific_file, "r") as f:
        specific_data = json.load(f)

    actionable_scores = {}

    for paper_name in generic_data:
        if paper_name in specific_data:
            generic_review = generic_data[paper_name]["review"]
            specific_review = specific_data[paper_name]["review"]

            generic_score = compute_actionable_score(generic_review)
            specific_score = compute_actionable_score(specific_review)

            actionable_scores[paper_name] = {
                "generic_actionable_verb_score": generic_score,
                "specific_actionable_verb_score": specific_score
            }

    return actionable_scores

generic_output_path = "C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\dtais_summer\\generic_iclr_100_1.json"
specific_output_path = "C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\dtais_summer\\specific_iclr_100_1.json"
# Example usage:
actionable_scores = compare_generic_specific(generic_output_path, specific_output_path)
for paper, scores in actionable_scores.items():
    print(f"{paper}: {scores}")


<>:6: SyntaxWarning: invalid escape sequence '\d'
<>:6: SyntaxWarning: invalid escape sequence '\d'
C:\Users\G34371231\AppData\Local\Temp\ipykernel_16228\4103925598.py:6: SyntaxWarning: invalid escape sequence '\d'
  """


606.pdf: {'generic_actionable_verb_score': 1.0, 'specific_actionable_verb_score': 0.0}
466.pdf: {'generic_actionable_verb_score': 1.0, 'specific_actionable_verb_score': 0.0}
602.pdf: {'generic_actionable_verb_score': 0.0, 'specific_actionable_verb_score': 1.0}
535.pdf: {'generic_actionable_verb_score': 0.0, 'specific_actionable_verb_score': 1.0}
548.pdf: {'generic_actionable_verb_score': 1.0, 'specific_actionable_verb_score': 1.0}
472.pdf: {'generic_actionable_verb_score': 1.0, 'specific_actionable_verb_score': 1.0}
681.pdf: {'generic_actionable_verb_score': 0.0, 'specific_actionable_verb_score': 1.0}
470.pdf: {'generic_actionable_verb_score': 1.0, 'specific_actionable_verb_score': 1.0}
550.pdf: {'generic_actionable_verb_score': 0.0, 'specific_actionable_verb_score': 0.0}
785.pdf: {'generic_actionable_verb_score': 0.0, 'specific_actionable_verb_score': 0.0}
394.pdf: {'generic_actionable_verb_score': 1.0, 'specific_actionable_verb_score': 1.0}
755.pdf: {'generic_actionable_verb_score': 

In [182]:
import json
import re
from nltk import word_tokenize, pos_tag

# Define actionable verbs
key_verbs = [
    # Core revision actions
    "add", "remove", "revise", "rewrite", "edit", "update", "reword", "rephrase",
    "elaborate", "shorten", "trim", "lengthen", "restructure", "reorganize",
    # Clarity and communication
    "clarify", "explain", "define", "specify", "illustrate", "highlight",
    "summarize", "emphasize", "articulate", "motivate", "paraphrase",
    # Argumentation and evidence
    "justify", "support", "strengthen", "back", "verify", "demonstrate",
    "validate", "critique", "challenge", "address", "acknowledge",
    # Comparison and context
    "compare", "contrast", "situate", "contextualize", "position", "relate",
    "connect", "differentiate", "link", "distinguish",
    # Presentation and organization
    "organize", "structure", "group", "separate", "move", "align", "combine",
    "merge", "categorize", "label", "format", "layout",
    # Content development
    "expand", "narrow", "introduce", "incorporate", "include", "exclude",
    "discuss", "mention", "eliminate", "outline", "detail", "cover",
    # Analysis and interpretation
    "analyze", "interpret", "evaluate", "examine", "investigate", "assess",
    "explore", "model", "test", "measure", "report", "simulate", "estimate",
    # Method and experimental critique
    "control", "describe", "replicate", "repeat", "redesign", "calibrate",
    "augment", "reimplement", "implement", "summarize",
    # Tone and style
    "tone down", "soften", "neutralize", "strengthen", "balance", "refine",
    "polish", "simplify", "formalize"
]
key_verbs_set = set(key_verbs)

def extract_numbered_suggestions(text):
    pattern = r"\n\d+\.\s+(.*?)(?=\n\d+\.|\Z)"
    matches = re.findall(pattern, text, re.DOTALL)
    return [m.strip() for m in matches if m.strip()]

def contains_verb(text):
    tokens = word_tokenize(text)
    tags = pos_tag(tokens)
    return any(tag.startswith("VB") for _, tag in tags)

def contains_actionable_keyword(text):
    tokens = [t.lower() for t in word_tokenize(text)]
    return any(token in key_verbs_set for token in tokens)

def compute_scores(review_text):
    suggestions = extract_numbered_suggestions(review_text)
    total = len(suggestions)
    
    # First method: any verb
    with_verb = sum(contains_verb(s) for s in suggestions)

    # Second method: verb from key list
    with_keyword_verb = sum(contains_actionable_keyword(s) for s in suggestions)

    return {
        "actionable_verb_score": round(with_verb / total, 2) if total else 0.0,
        "actionable_keyword_verb_score": round(with_keyword_verb / total, 2) if total else 0.0
    }

def compare_generic_specific(generic_file, specific_file):
    with open(generic_file, "r") as f:
        generic_data = json.load(f)
    with open(specific_file, "r") as f:
        specific_data = json.load(f)

    actionable_scores = {}

    for paper_name in generic_data:
        if paper_name in specific_data:
            generic_review = generic_data[paper_name]["review"]
            specific_review = specific_data[paper_name]["review"]

            generic_scores = compute_scores(generic_review)
            specific_scores = compute_scores(specific_review)

            actionable_scores[paper_name] = {
                "generic_actionable_verb_score": generic_scores["actionable_verb_score"],
                "specific_actionable_verb_score": specific_scores["actionable_verb_score"],
                "generic_actionable_keyword_verb_score": generic_scores["actionable_keyword_verb_score"],
                "specific_actionable_keyword_verb_score": specific_scores["actionable_keyword_verb_score"]
            }

    return actionable_scores

# Example usage:
actionable_scores = compare_generic_specific(generic_output_path, specific_output_path)

# Print summary
for paper, scores in actionable_scores.items():
    print(f"{paper}: {scores}")


606.pdf: {'generic_actionable_verb_score': 1.0, 'specific_actionable_verb_score': 0.0, 'generic_actionable_keyword_verb_score': 0.62, 'specific_actionable_keyword_verb_score': 0.0}
466.pdf: {'generic_actionable_verb_score': 1.0, 'specific_actionable_verb_score': 0.0, 'generic_actionable_keyword_verb_score': 0.3, 'specific_actionable_keyword_verb_score': 0.0}
602.pdf: {'generic_actionable_verb_score': 0.0, 'specific_actionable_verb_score': 1.0, 'generic_actionable_keyword_verb_score': 0.0, 'specific_actionable_keyword_verb_score': 0.6}
535.pdf: {'generic_actionable_verb_score': 0.0, 'specific_actionable_verb_score': 1.0, 'generic_actionable_keyword_verb_score': 0.0, 'specific_actionable_keyword_verb_score': 0.5}
548.pdf: {'generic_actionable_verb_score': 1.0, 'specific_actionable_verb_score': 1.0, 'generic_actionable_keyword_verb_score': 0.6, 'specific_actionable_keyword_verb_score': 0.88}
472.pdf: {'generic_actionable_verb_score': 1.0, 'specific_actionable_verb_score': 1.0, 'generic_ac

In [183]:
import json
import re
from nltk import word_tokenize, pos_tag
from nltk.corpus import wordnet as wn

# Define actionable verbs
key_verbs = [
    # Core revision actions
    "add", "remove", "revise", "rewrite", "edit", "update", "reword", "rephrase",
    "elaborate", "shorten", "trim", "lengthen", "restructure", "reorganize",
    # Clarity and communication
    "clarify", "explain", "define", "specify", "illustrate", "highlight",
    "summarize", "emphasize", "articulate", "motivate", "paraphrase",
    # Argumentation and evidence
    "justify", "support", "strengthen", "back", "verify", "demonstrate",
    "validate", "critique", "challenge", "address", "acknowledge",
    # Comparison and context
    "compare", "contrast", "situate", "contextualize", "position", "relate",
    "connect", "differentiate", "link", "distinguish",
    # Presentation and organization
    "organize", "structure", "group", "separate", "move", "align", "combine",
    "merge", "categorize", "label", "format", "layout",
    # Content development
    "expand", "narrow", "introduce", "incorporate", "include", "exclude",
    "discuss", "mention", "eliminate", "outline", "detail", "cover",
    # Analysis and interpretation
    "analyze", "interpret", "evaluate", "examine", "investigate", "assess",
    "explore", "model", "test", "measure", "report", "simulate", "estimate",
    # Method and experimental critique
    "control", "describe", "replicate", "repeat", "redesign", "calibrate",
    "augment", "reimplement", "implement", "summarize",
    # Tone and style
    "tone down", "soften", "neutralize", "strengthen", "balance", "refine",
    "polish", "simplify", "formalize"
]
key_verbs_set = set(key_verbs)

def extract_numbered_suggestions(text):
    pattern = r"\n\d+\.\s+(.*?)(?=\n\d+\.|\Z)"
    matches = re.findall(pattern, text, re.DOTALL)
    return [m.strip() for m in matches if m.strip()]

def contains_verb(text):
    tokens = word_tokenize(text)
    tags = pos_tag(tokens)
    return any(tag.startswith("VB") for _, tag in tags)

def contains_actionable_keyword(text):
    tokens = [t.lower() for t in word_tokenize(text)]
    return any(token in key_verbs_set for token in tokens)

def get_wordnet_depth(verb):
    synsets = wn.synsets(verb, pos=wn.VERB)
    if not synsets:
        return 0
    return max(synset.max_depth() for synset in synsets)

def average_verb_depth(text_blocks):
    depths = []
    for block in text_blocks:
        tokens = word_tokenize(block)
        pos_tags = pos_tag(tokens)
        block_depths = []

        for word, tag in pos_tags:
            if tag.startswith("VB"):
                depth = get_wordnet_depth(word.lower())
                if depth > 0:
                    block_depths.append(depth)

        if block_depths:
            depths.append(max(block_depths))  # Take the most specific verb in the block

    if depths:
        return round(sum(depths) / len(depths), 2)
    return 0.0

def compute_scores(review_text):
    suggestions = extract_numbered_suggestions(review_text)
    total = len(suggestions)

    with_verb = sum(contains_verb(s) for s in suggestions)
    with_keyword_verb = sum(contains_actionable_keyword(s) for s in suggestions)
    avg_verb_depth = average_verb_depth(suggestions)

    return {
        "actionable_verb_score": round(with_verb / total, 2) if total else 0.0,
        "actionable_keyword_verb_score": round(with_keyword_verb / total, 2) if total else 0.0,
        "average_verb_depth_score": avg_verb_depth
    }

def compare_generic_specific(generic_file, specific_file):
    with open(generic_file, "r") as f:
        generic_data = json.load(f)
    with open(specific_file, "r") as f:
        specific_data = json.load(f)

    actionable_scores = {}

    for paper_name in generic_data:
        if paper_name in specific_data:
            generic_review = generic_data[paper_name]["review"]
            specific_review = specific_data[paper_name]["review"]

            generic_scores = compute_scores(generic_review)
            specific_scores = compute_scores(specific_review)

            actionable_scores[paper_name] = {
                "generic_actionable_verb_score": generic_scores["actionable_verb_score"],
                "specific_actionable_verb_score": specific_scores["actionable_verb_score"],
                "generic_actionable_keyword_verb_score": generic_scores["actionable_keyword_verb_score"],
                "specific_actionable_keyword_verb_score": specific_scores["actionable_keyword_verb_score"],
                "generic_verb_depth_score": generic_scores["average_verb_depth_score"],
                "specific_verb_depth_score": specific_scores["average_verb_depth_score"]
            }

    return actionable_scores

# Example usage:
actionable_scores = compare_generic_specific(generic_output_path, specific_output_path)

for paper, scores in actionable_scores.items():
    print(f"{paper}: {scores}")


606.pdf: {'generic_actionable_verb_score': 1.0, 'specific_actionable_verb_score': 0.0, 'generic_actionable_keyword_verb_score': 0.62, 'specific_actionable_keyword_verb_score': 0.0, 'generic_verb_depth_score': 8.5, 'specific_verb_depth_score': 0.0}
466.pdf: {'generic_actionable_verb_score': 1.0, 'specific_actionable_verb_score': 0.0, 'generic_actionable_keyword_verb_score': 0.3, 'specific_actionable_keyword_verb_score': 0.0, 'generic_verb_depth_score': 7.1, 'specific_verb_depth_score': 0.0}
602.pdf: {'generic_actionable_verb_score': 0.0, 'specific_actionable_verb_score': 1.0, 'generic_actionable_keyword_verb_score': 0.0, 'specific_actionable_keyword_verb_score': 0.6, 'generic_verb_depth_score': 0.0, 'specific_verb_depth_score': 9.0}
535.pdf: {'generic_actionable_verb_score': 0.0, 'specific_actionable_verb_score': 1.0, 'generic_actionable_keyword_verb_score': 0.0, 'specific_actionable_keyword_verb_score': 0.5, 'generic_verb_depth_score': 0.0, 'specific_verb_depth_score': 9.0}
548.pdf: {'

In [184]:
from tabulate import tabulate

def compute_overall_average_scores(actionable_scores):
    totals = {
        "generic_actionable_verb_score": 0.0,
        "specific_actionable_verb_score": 0.0,
        "generic_actionable_keyword_verb_score": 0.0,
        "specific_actionable_keyword_verb_score": 0.0,
    }

    for scores in actionable_scores.values():
        for key in totals:
            totals[key] += scores[key]

    # Divide each total by 100 (assumes 100 papers)
    averages = {key: round(totals[key] / 100, 3) for key in totals}
    return averages

# Example usage
overall_averages = compute_overall_average_scores(actionable_scores)
print("Overall Actionability Scores (averaged across 100 papers):")
for k, v in overall_averages.items():
    print(f"{k}: {v}")

# Format into a table
table = [
    [
        "Verb Test",
        overall_averages["generic_actionable_verb_score"],
        overall_averages["specific_actionable_verb_score"]
    ],
    [
        "Keyword Verb Test",
        overall_averages["generic_actionable_keyword_verb_score"],
        overall_averages["specific_actionable_keyword_verb_score"]
    ]
]

headers = ["", "Generic Prompt", "Specific Prompt"]
print("\n Average Actionability Scores for 100 ICLR Papers:\n")
print(tabulate(table, headers=headers, floatfmt=".3f"))



Overall Actionability Scores (averaged across 100 papers):
generic_actionable_verb_score: 0.847
specific_actionable_verb_score: 0.869
generic_actionable_keyword_verb_score: 0.485
specific_actionable_keyword_verb_score: 0.519

 Average Actionability Scores for 100 ICLR Papers:

                     Generic Prompt    Specific Prompt
-----------------  ----------------  -----------------
Verb Test                     0.847              0.869
Keyword Verb Test             0.485              0.519


In [185]:
from tabulate import tabulate

def compute_overall_average_scores(actionable_scores):
    totals = {
        "generic_actionable_verb_score": 0.0,
        "specific_actionable_verb_score": 0.0,
        "generic_actionable_keyword_verb_score": 0.0,
        "specific_actionable_keyword_verb_score": 0.0,
        "generic_verb_depth_score": 0.0,
        "specific_verb_depth_score": 0.0,
    }

    num_files = len(actionable_scores)

    for scores in actionable_scores.values():
        for key in totals:
            totals[key] += scores[key]

    averages = {
        key: round(totals[key] / num_files, 3) if num_files > 0 else 0.0
        for key in totals
    }
    return averages

# Example usage
overall_averages = compute_overall_average_scores(actionable_scores)

print("Overall Actionability Scores (averaged across all papers):")
for k, v in overall_averages.items():
    print(f"{k}: {v}")

# Format into a table
table = [
    [
        "Verb Test",
        overall_averages["generic_actionable_verb_score"],
        overall_averages["specific_actionable_verb_score"]
    ],
    [
        "Keyword Verb Test",
        overall_averages["generic_actionable_keyword_verb_score"],
        overall_averages["specific_actionable_keyword_verb_score"]
    ],
    [
        "Verb Depth Score",
        overall_averages["generic_verb_depth_score"],
        overall_averages["specific_verb_depth_score"]
    ]
]

headers = ["", "Generic Prompt", "Specific Prompt"]
print("\nAverage Actionability Scores Across Papers:\n")
print(tabulate(table, headers=headers, floatfmt=".3f"))


Overall Actionability Scores (averaged across all papers):
generic_actionable_verb_score: 0.847
specific_actionable_verb_score: 0.869
generic_actionable_keyword_verb_score: 0.485
specific_actionable_keyword_verb_score: 0.519
generic_verb_depth_score: 6.864
specific_verb_depth_score: 7.113

Average Actionability Scores Across Papers:

                     Generic Prompt    Specific Prompt
-----------------  ----------------  -----------------
Verb Test                     0.847              0.869
Keyword Verb Test             0.485              0.519
Verb Depth Score              6.864              7.113


In [186]:
from nltk.corpus import wordnet as wn

def print_verb_hypernym_path(verb):
    synsets = wn.synsets(verb, pos=wn.VERB)
    if not synsets:
        print(f"No verb synsets found for: {verb}")
        return

    syn = synsets[0]  # Take the most common verb sense
    print(f"🔹 Synset: {syn.name()}")
    print(f"   Definition: {syn.definition()}\n")

    paths = syn.hypernym_paths()
    if not paths:
        print("No hypernym paths found.")
        return

    print("🔗 Hypernym Path (to root):")
    for depth, node in enumerate(paths[0]):
        indent = "  " * depth
        print(f"{indent}- {node.name()} — {node.definition()}")


In [187]:
print_verb_hypernym_path('clarify')

🔹 Synset: clarify.v.01
   Definition: make clear and (more) comprehensible

🔗 Hypernym Path (to root):
- act.v.01 — perform an action, or work out or perform (an action)
  - interact.v.01 — act together or towards others or with others
    - communicate.v.02 — transmit thoughts or feelings
      - inform.v.01 — impart knowledge of some fact, state or affairs, or event to
        - explain.v.01 — make plain and comprehensible
          - clarify.v.01 — make clear and (more) comprehensible


In [188]:
print_verb_hypernym_path('evaluate')

🔹 Synset: measure.v.04
   Definition: evaluate or estimate the nature, quality, ability, extent, or significance of

🔗 Hypernym Path (to root):
- think.v.03 — use or exercise the mind or one's power of reason in order to make inferences, decisions, or arrive at a solution or judgments
  - evaluate.v.02 — form a critical opinion of
    - measure.v.04 — evaluate or estimate the nature, quality, ability, extent, or significance of


In [189]:
print_verb_hypernym_path('Consider')

🔹 Synset: see.v.05
   Definition: deem to be

🔗 Hypernym Path (to root):
- think.v.03 — use or exercise the mind or one's power of reason in order to make inferences, decisions, or arrive at a solution or judgments
  - evaluate.v.02 — form a critical opinion of
    - think.v.01 — judge or regard; look upon; judge
      - see.v.05 — deem to be
